# Phase 1 - AIST++ SMPL 24-Joint Validation

This notebook validates the Phase 1 data path on Kaggle: AIST++ SMPL loading, batching to `[B, T, 72]`, and 3D skeleton rendering. It does not train the diffusion model.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Dependencies installed")

In [ ]:
from pathlib import Path
import os
import yaml
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Torch device: {device}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

def detect_aist_root() -> Path:
    if os.environ.get("AISTPP_ROOT"):
        return Path(os.environ["AISTPP_ROOT"]).expanduser()
    input_root = Path("/kaggle/input")
    if not input_root.exists():
        raise FileNotFoundError("/kaggle/input not found. Set AISTPP_ROOT to your local AIST++ root.")
    for suffix in ("*.pkl", "*.npz"):
        for path in input_root.glob(f"**/{suffix}"):
            parts = path.parts
            if len(parts) >= 4 and parts[1] == "kaggle" and parts[2] == "input":
                return Path(*parts[:4])
    raise FileNotFoundError("No .pkl/.npz AIST++ motion files found under /kaggle/input.")

aist_root = detect_aist_root()
print(f"Using AIST++ root: {aist_root}")

In [ ]:
with open("config.yaml", "r", encoding="utf-8") as handle:
    raw_config = yaml.safe_load(handle)

raw_config["project"]["output_dir"] = "outputs_phase1_aist"
raw_config["data"]["source"] = "aistpp"
raw_config["data"]["representation"] = "smpl_axis_angle_24x3"
raw_config["data"]["num_joints"] = 24
raw_config["data"]["input_dim"] = 72
raw_config["data"]["sequence_length"] = min(int(raw_config["data"]["sequence_length"]), 240)
raw_config["data"]["batch_size"] = 8
raw_config["data"]["aist"]["root_dir"] = str(aist_root)
raw_config["data"]["aist"]["split_dir"] = str(aist_root / "splits") if (aist_root / "splits").exists() else raw_config["data"]["aist"].get("split_dir")
raw_config["data"]["aist"]["ignore_list_path"] = str(aist_root / "ignore_list.txt") if (aist_root / "ignore_list.txt").exists() else raw_config["data"]["aist"].get("ignore_list_path")
raw_config["data"]["aist"]["max_files_per_split"] = 24
raw_config["visualization"]["max_frames"] = 120
raw_config["visualization"]["fps"] = 24
raw_config["model"]["input_dim"] = 72

phase_config_path = Path("outputs_phase1_aist/phase1_config.yaml")
phase_config_path.parent.mkdir(parents=True, exist_ok=True)
with open(phase_config_path, "w", encoding="utf-8") as handle:
    yaml.safe_dump(raw_config, handle, sort_keys=False)

print(f"Wrote validation config: {phase_config_path}")

In [ ]:
from embodied_motion_flow.config import load_config
from embodied_motion_flow.reproducibility import set_global_seed
from embodied_motion_flow.data.dataset import build_dataloaders

config = load_config(phase_config_path)
set_global_seed(
    seed=config.reproducibility.seed,
    deterministic_torch=config.reproducibility.deterministic_torch,
    benchmark_cudnn=config.reproducibility.benchmark_cudnn,
)

splits = build_dataloaders(config)
print(f"Train clips: {len(splits.train_dataset)}")
print(f"Val clips:   {len(splits.val_dataset)}")
print(f"Test clips:  {len(splits.test_dataset)}")

batch = next(iter(splits.train_loader))
motion = batch["motion"]
print(f"Batch motion shape: {tuple(motion.shape)}")
assert motion.ndim == 3
assert motion.shape[-1] == 72
print("AIST++ SMPL batch validated: [B, T, 72]")

In [ ]:
from IPython.display import Image, Video, display
from embodied_motion_flow.visualization.smpl_render import save_smpl_static_frame, save_smpl_3d_animation

sample_motion = motion[0].detach().cpu().numpy()
plot_dir = Path(config.project.output_dir) / "plots"
animation_dir = Path(config.project.output_dir) / "animations"
static_path = save_smpl_static_frame(sample_motion, plot_dir / "smpl_static_frame.png", frame_index=0, dpi=config.visualization.dpi)
video_path = save_smpl_3d_animation(
    sample_motion,
    animation_dir / "smpl_3d_motion.mp4",
    fps=config.visualization.fps,
    max_frames=config.visualization.max_frames,
    dpi=config.visualization.dpi,
)

display(Image(filename=str(static_path)))
display(Video(filename=str(video_path), embed=True))
print(f"Saved static frame: {static_path}")
print(f"Saved SMPL MP4: {video_path}")